# 국토부 ARS로 STCIS 정류장 ID 조회

07번에서 사용한 ARS 조회 방식처럼, 이름 기준으로 만든 국토부 후보 CSV의 ARS를 STCIS `bussttn` API에 조회한다.
요청은 200개씩 나누고 요청 사이에 충분한 간격을 둔다. 각 요청 직후 체크포인트를 저장하므로 중간에 연결이 끊겨도 체크포인트 파일을 이용해 이어서 실행할 수 있다.

### 셀 1. 국토부 정류장 위치 CSV에서 ARS 후보 준비\n

### ?? ? 1. ??? ?????? ARS ??? ??


In [ ]:
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
INPUT = DATA_DIR / 'route_name_ars_reference_2200_20241017.csv'
DATE = '20241017'
ENDPOINT = 'https://stcis.go.kr/openapi/bussttn.json'
API_KEY = getpass('STCIS 인증키를 입력하세요: ').strip()
if not API_KEY:
    raise ValueError('인증키가 입력되지 않았습니다.')
if not INPUT.exists():
    raise FileNotFoundError(f'입력 파일이 없습니다: {INPUT}')

# 200개 단위 분할 및 과부하 방지 간격
CHUNK_SIZE = 200
REQUEST_SLEEP = 2.0
CHUNK_SLEEP = 15.0
TIMEOUT = 45

candidate = pd.read_csv(INPUT, dtype=str, encoding='utf-8-sig').fillna('')
candidate['ARS'] = candidate['ARS'].astype(str).str.strip()
candidate = candidate[candidate['ARS'].ne('') & candidate['ARS'].ne('0')].copy()

# 국토부 파일의 도시명으로 시도코드를 결정한다.
candidate['sdCd'] = candidate['도시명'].map(
    lambda x: '11' if '서울' in str(x) else ('41' if '경기' in str(x) else '')
)
candidate = candidate[candidate['sdCd'].ne('')].copy()
candidate['ARS'] = candidate['ARS'].str.replace(r'\.0$', '', regex=True).str.zfill(5)
candidate = candidate.rename(columns={'노선번호': 'routeNo', '순서': 'sttnSeq', '정류장명': 'sttnNm'})
targets = candidate[['routeNo', 'sttnSeq', 'sttnNm', 'ARS', 'sdCd']].drop_duplicates().reset_index(drop=True)
# 이전 07/08 조회 결과에 이미 있는 ARS는 다시 요청하지 않는다.
previous = DATA_DIR / f'ars_stop_id_lookup_reference_{DATE}.csv'
if previous.exists():
    old = pd.read_csv(previous, dtype=str, encoding='utf-8-sig').fillna('')
    old_ars_col = '조회ARS' if '조회ARS' in old.columns else ('sttnArsno' if 'sttnArsno' in old.columns else '')
    if old_ars_col:
        targets = targets[~targets['ARS'].isin(old[old_ars_col].str.zfill(5))].reset_index(drop=True)
print(f'전체 후보: {len(targets):,}개 / 청크 크기: {CHUNK_SIZE}개')


### 셀 2. 2200번 ARS로 STCIS 정류장 ID 조회\n

### ?? ? 2. 2200? ARS? STCIS ??? ID ?? ? ????? ??


In [ ]:
RESULT = DATA_DIR / f'ars_stop_id_lookup_reference_2200_{DATE}.csv'
CHECKPOINT = DATA_DIR / f'ars_stop_id_lookup_reference_2200_{DATE}_checkpoint.csv'

def parse_payload(payload):
    result = payload.get('result', [])
    if isinstance(result, dict):
        result = [result]
    if not isinstance(result, list):
        result = []
    return result, str(payload.get('status', ''))

# 기존 체크포인트가 있으면 성공적으로 저장된 ARS는 건너뛴다.
if CHECKPOINT.exists():
    saved = pd.read_csv(CHECKPOINT, dtype=str, encoding='utf-8-sig').fillna('')
else:
    saved = pd.DataFrame()

done_keys = set()
if not saved.empty:
    ok = saved[saved.get('조회상태', '').eq('OK')] if '조회상태' in saved.columns else saved
    done_keys = set(zip(ok.get('조회ARS', []), ok.get('sdCd', [])))
pending = targets[~targets.apply(lambda r: (r['ARS'], r['sdCd']) in done_keys, axis=1)].reset_index(drop=True)
print(f'기존 완료: {len(done_keys):,}개 / 이번 조회: {len(pending):,}개')

session = requests.Session()
records = saved.to_dict('records') if not saved.empty else []
chunks = [pending.iloc[i:i + CHUNK_SIZE] for i in range(0, len(pending), CHUNK_SIZE)]

for chunk_no, chunk in enumerate(chunks, start=1):
    print(f'\n===== 청크 {chunk_no}/{len(chunks)} ({len(chunk)}개) =====')
    for _, row in chunk.iterrows():
        params = {
            'apikey': API_KEY,
            'sdCd': row['sdCd'],
            'sttnArsno': row['ARS'],
        }
        base = {
            'routeNo': row['routeNo'],
            'sttnSeq': row['sttnSeq'],
            'source_sttnNm': row['sttnNm'],
            '조회ARS': row['ARS'],
            'sdCd': row['sdCd'],
        }
        try:
            response = session.get(ENDPOINT, params=params, timeout=TIMEOUT)
            response.raise_for_status()
            items, status = parse_payload(response.json())
            if items:
                for item in items:
                    record = dict(base)
                    record.update(item)
                    record['조회상태'] = status or 'OK'
                    records.append(record)
            else:
                record = dict(base)
                record['조회상태'] = status or 'NOT_FOUND'
                records.append(record)
            print(f"{row['routeNo']} / {row['ARS']}: {len(items)}건")
        except Exception as exc:
            record = dict(base)
            record['조회상태'] = 'ERROR'
            record['error'] = repr(exc)
            records.append(record)
            print(f"{row['routeNo']} / {row['ARS']}: 실패 - {exc}")
        # 요청 하나가 끝날 때마다 즉시 저장
        pd.DataFrame(records).to_csv(CHECKPOINT, index=False, encoding='utf-8-sig')
        time.sleep(REQUEST_SLEEP)
    pd.DataFrame(records).to_csv(CHECKPOINT, index=False, encoding='utf-8-sig')
    if chunk_no < len(chunks):
        print(f'청크 사이 {CHUNK_SLEEP:.0f}초 대기')
        time.sleep(CHUNK_SLEEP)

result = pd.DataFrame(records)
result.to_csv(RESULT, index=False, encoding='utf-8-sig')
print(f'최종 저장: {RESULT} ({len(result):,}행)')
print(f'체크포인트: {CHECKPOINT}')


### 셀 3. 9030·9030-1 제공 ARS로 정류장 ID 역조회\n

### ?? ? 3. 9030?9030-1? ?? ARS? STCIS ??? ID? ??


In [ ]:
# 9030·9030-1 전용: 제공된 ARS로 정류장 ID 역조회
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
ENDPOINT = 'https://stcis.go.kr/openapi/bussttn.json'
API_KEY = getpass('STCIS 인증키를 입력하세요: ').strip()
if not API_KEY: raise ValueError('인증키가 입력되지 않았습니다.')

rows = [
    ('9030',1,'교하차고지','31891'),('9030',2,'오도1리','30152'),('9030',3,'오도리','31769'),('9030',4,'오도삼거리','31412'),
    ('9030',5,'다락골','30671'),('9030',6,'다율동','30701'),('9030',7,'난방공사','30682'),('9030',8,'청석마을8단지','30644'),
    ('9030',9,'청석마을9단지','30684'),('9030',10,'숲속길마을7단지','30646'),('9030',11,'숲속길마을3.6단지','30673'),('9030',12,'물향기마을1단지','31974'),
    ('9030',13,'해오름마을중심상가','31868'),('9030',14,'산내마을6.8단지','30103'),('9030',15,'산내마을8.12단지(중)','31766'),('9030',16,'두레공원(중)','31672'),
    ('9030',17,'해솔마을4단지','30148'),('9030',18,'지산중학교.운정행복센터','31673'),('9030',19,'해솔마을2.5단지','31676'),('9030',20,'산내마을11단지','31677'),
    ('9030',21,'운정고등학교','31696'),('9030',22,'한울마을1단지.LH파주본부','31653'),('9030',23,'한울마을2단지','31684'),('9030',24,'경기인력개발원','30823'),('9030',25,'새암공원','31628'),
    ('9030-1',1,'교하차고지','31825'),('9030-1',2,'가람마을3.4.6단지','31701'),('9030-1',3,'가람마을2.8단지(중)','31669'),('9030-1',4,'두레공원(중)','31670'),
    ('9030-1',5,'산내마을8.12단지(중)','31765'),('9030-1',6,'산내마을6.8단지','30104'),('9030-1',7,'해오름마을중심상가','31966'),('9030-1',8,'숲속길마을7단지','30646'),
    ('9030-1',9,'숲속길마을3.6단지','30673'),('9030-1',10,'문발동.아랫말','30916'),('9030-1',11,'출판단지삼거리','31726'),('9030-1',12,'돌곶이꽃마을','30943'),
    ('9030-1',13,'롯데프리미엄아울렛','31709'),('9030-1',14,'심학교','30923'),('9030-1',15,'이채쇼핑몰','30006'),('9030-1',16,'이석사거리','30680'),
    ('9030-1',17,'은석교사거리','30003'),('9030-1',18,'북센후문','31572'),('9030-1',19,'북센정문','31571'),('9030-1',20,'송운사','31573')
]
targets = pd.DataFrame(rows, columns=['routeNo','sttnSeq','source_sttnNm','ARS'])
targets['ARS'] = targets['ARS'].astype(str).str.zfill(5)
records = []
session = requests.Session()
for i, row in targets.iterrows():
    params = {'apikey': API_KEY, 'sdCd': '41', 'sttnArsno': row['ARS']}
    base = row.to_dict()
    try:
        response = session.get(ENDPOINT, params=params, timeout=45)
        response.raise_for_status()
        payload = response.json()
        items = payload.get('result', [])
        if isinstance(items, dict): items = [items]
        if not isinstance(items, list): items = []
        if items:
            for item in items:
                record = {**base, **item, '조회상태': payload.get('status','OK')}
                records.append(record)
        else:
            records.append({**base, '조회상태': payload.get('status','NOT_FOUND')})
        print(f"[{i+1}/{len(targets)}] {row['routeNo']} / {row['ARS']}: {len(items)}건")
    except Exception as exc:
        records.append({**base, '조회상태': 'ERROR', 'error': repr(exc)})
        print(f"[{i+1}/{len(targets)}] {row['routeNo']} / {row['ARS']}: 실패 - {exc}")
    time.sleep(2.0)

output = DATA_DIR / 'ars_stop_id_lookup_reference_9030_9030-1_20241017.csv'
pd.DataFrame(records).to_csv(output, index=False, encoding='utf-8-sig')
print(f'9030·9030-1 ARS 조회 결과 저장: {output} ({len(records)}건)')